<a href="https://colab.research.google.com/github/kanebako-s/RealEstate_analysis/blob/main/RealEstate_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 分析の目的

「国が認めた価値（公示地価）」と「市場が動いた結果（取引価格）」のズレから、地域の変化を浮き彫りにすることで、過熱エリアと過小評価エリアを導き出します

In [1]:
# --- ライブラリのインポート　---
import pandas as pd
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from pathlib import Path

# --- データの表示設定 ---
pd.options.display.max_columns = None # 列数が多い時、省略（...）せずに全部表示する
pd.set_option('display.float_format', '{:.2f}'.format) # 小数点以下を2桁に固定して見やすくする

# --- 日本語化（重要！） ---
!pip install japanize-matplotlib
import japanize_matplotlib # これでグラフの日本語が「□」にならずに済む

# --- Worldcloudのインストール ---
!pip install wordcloud
from wordcloud import WordCloud

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 45.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for japanize-matplotlib: filename=japanize_matplotlib-1.1.3-py3-none-any.whl size=4120257 sha256=9093dc471a2d345c35d207a25da7af8aab803249656c7e048f03acbae06fcf4c
  Stored in directory: /root/.cache/pip/wheels/c1/f7/9b/418f19a7b9340fc16e071e89efc379aca68d40238b258df53d
Successfully built japanize-matplotlib


分析の土台となるファイルの読み込み :

In [2]:
# パスを定義
PROJECT_ROOT = Path("/content/drive/MyDrive/RealEstate_DateAnalist_Learning")
GOV_DATA_BASE = PROJECT_ROOT / "hyokasyo-2026-東京都 2"
KAGGLE_DATA_BASE = PROJECT_ROOT / "archive-2"

# ファイルの読み込み
df_gov = pd.read_csv(
    GOV_DATA_BASE / "2026_TAKUCHI_k_13.csv",
    encoding='cp932',
    header=None,
    dtype=str  # 全て文字列で読み込む安全策
)
df_kaggle = pd.read_csv(
    KAGGLE_DATA_BASE / "Kaggle_All_prefectures_buildings_with_migration.csv",
    dtype=str  # 全て文字列で読み込む安全策
)

In [3]:
df_kaggle.shape

(1339147, 32)

In [4]:
df_gov.shape

(5104, 1408)

### 国土交通省の地価公示データの前処理

本ステップでは、1400項目を超える国土交通省の地価公示データから、分析の目的に合致する主要な5項目を選出した。公式のレイアウト定義書（PDF）が不在の状況下であったが、以下の特定プロセスを経て列の情報を確定させた。

1. 「千代田区」「〜町〜番地」といった既知の地名や住所が記されていることから 列3と列26は「市区町村」「住所」と推測しました。

2. 住所からその土地の平米単価をgoogle検索をして、その数値と近似している列を特定し、列19が「平米単価」と推測しました。

3. 住所それぞれの最寄駅を調べ一致している列を特定した。したがって列49は「最寄駅」と推測しました。

4. 住所と最寄り駅の距離を一件づつ調べ、その数値と各列の持つ数値を照らし合わせました。そして「駅まで何mか」という情報を持つ列を列50と推測しました

In [5]:
# 分析に使用する列を選出
df_selected = df_gov.iloc[:, [3, 19, 26, 49, 50]].copy()

# 抽出した列に名前をつける（日本語でOK、後のグラフ作成が楽になります）
df_selected.columns = ['市区町村', '公示地価', '住所', '最寄り駅', '駅距離']

# 公示地価と駅距離を数値型に変換
# errors='coerce' をつけることで、万が一数値以外の文字が入っていてもエラーにならず「欠損値(NaN)」として処理してくれます
df_selected['公示地価'] = pd.to_numeric(df_selected['公示地価'], errors='coerce')
df_selected['駅距離'] = pd.to_numeric(df_selected['駅距離'], errors='coerce')

In [6]:
print(df_selected.describe())

             公示地価     駅距離
count     5104.00 5104.00
mean   1483484.48  829.53
std    3847920.76  882.29
min       5400.00    0.00
25%     291750.00  300.00
50%     563000.00  600.00
75%    1180000.00 1100.00
max   67600000.00 9100.00


結果 : 特定した公示地価と駅距離は不動産市場の常識的に妥当である。

In [7]:
# 欠損値の数を確認
print(df_selected.isnull().sum())

市区町村    0
公示地価    0
住所      0
最寄り駅    0
駅距離     0
dtype: int64


洞察 : 政府のデータは情報が一緒くたになっているだけで書式は統一されていてきれいなのかもしれない。

本ステップでは土地の「用途区分コード（住宅地、商業地、工業地、宅地見込地）」と上物がある場合の「築年数」の情報を抽出します。

** 用途区分コードの特定 **

仮説 : 用途区分コードは都市建設法の基づいて割り当てられるため種類数は2〜15種類程度に収まるはずである。

In [8]:
# 商業地の代表（中央区銀座4丁目：山野楽器付近など）
ginza_check = df_gov[df_gov.iloc[:, 26].str.contains('銀座４丁目', na=False)].iloc[:, [26, 31, 35]]

# 住宅地の代表（千代田区三番町）
sanbancho_check = df_gov[df_gov.iloc[:, 26].str.contains('三番町', na=False)].iloc[:, [26, 31, 35]]

print("--- 銀座（商業地）の列31 ---")
print(ginza_check.head(3))

print("\n--- 三番町（住宅地）の列31 ---")
print(sanbancho_check.head(3))

--- 銀座（商業地）の列31 ---
              26 31 35
168  銀座４丁目１０３番１外  7  1
169  銀座４丁目１０３番１外  7  1
176     銀座４丁目２番４  3  3

--- 三番町（住宅地）の列31 ---
        26 31 35
0  三番町６番２５  3  1
1  三番町６番２５  3  1


### 日本の住宅取引額データ（2005年〜2024年）の前処理

本ステップでは国土交通省の地価公示データと揃えるために、Kaggleデータから「東京都エリアのデータ」かつ「2024年のデータ」「分析に使用する列」を抽出します。

* 表記揺れ（大文字小文字の混在）や予期せぬ欠損値による処理の中断を防ぐ構造にし、目的の『東京都』データを網羅的に抽出できるようにしました。

* 東京エリアのデータを抽出後、迅速に前処理ロジックを固めるために、1万件のサンプルデータを用いて前処理パイプラインのプロトタイプを構築します。


In [17]:
# 2024年の東京都エリアのデータを抽出
df_Tokyo = df_kaggle[df_kaggle["Prefecture"].str.contains("Tokyo", case=False, na=False)]
df_Tokyo['Year'] = pd.to_numeric(df_Tokyo['Year'], errors='coerce')
df_Tokyo = df_Tokyo[df_Tokyo['Year'].astype(str) == '2024']

# 扱うデータ件数の最大値を決める。
population_size = len(df_Tokyo)  # 現在のデータ件数を取得
sample_size = min(10000, population_size)  # 1万件と現在の件数のうち、小さい方を採用

# df_Tokyoから最大で1万行だけ抽出
df_Tokyo_sample = df_Tokyo.sample(n=sample_size, random_state=42)

# インデックスをリセット
df_Tokyo_sample = df_Tokyo_sample.reset_index(drop=True)

# 分析に使用する列を抽出('Type'は住宅用地の一種類だったため今回は除外しました)
df_Tokyo_sample = df_Tokyo_sample[['City,Town,Ward,Village code', 'Year', 'TotalTransactionValue', 'ConstructionYear', 'BuildingCoverageRatio', 'FloorAreaRatio', 'AverageTimeToStation']]

# 列名を分かりやすい日本語に変換する辞書（定義書）
rename_dict = {
    'City,Town,Ward,Village code': '市区町村コード',
    'Year': '取引年',
    'TotalTransactionValue': '取引総額',
    'ConstructionYear': '建築年',
    'BuildingCoverageRatio': '建ぺい率',
    'FloorAreaRatio': '容積率',
    'AverageTimeToStation': '駅徒歩分'
}

# 列名を書き換える
df_Tokyo_sample = df_Tokyo_sample.rename(columns=rename_dict)


# データの最初の5行を表示して確認
print(df_Tokyo_sample.head())

  市区町村コード   取引年      取引総額   建築年  建ぺい率    容積率 駅徒歩分
0   13218  2024  40000000  2024  80.0  300.0    2
1   13201  2024  46000000  2024  40.0   80.0   12
2   13119  2024  72000000  2024  60.0  200.0   13
3   13110  2024  90000000  2024  60.0  150.0    7
4   13223  2024  42000000  2024  40.0   80.0   12


/tmp/ipykernel_870/617586963.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_Tokyo['Year'] = pd.to_numeric(df_Tokyo['Year'], errors='coerce')


本ステップでは'df_Tokyo_sample'の各カラムに対して、後続のSQL集計やデータ結合（JOIN）に備えたデータクレンジングを行います。

* 国交省側のデータとデータ型を統一し、「余白ゴミ」を洗い落としました。

* '駅徒歩分'に欠損値（NaN）があった場合、少数型（`.0`）ににhに変身してしまうことも考慮して整数型になるように処理しました。

In [19]:
# 市区町村コードのデータ型の統一
df_Tokyo_sample['市区町村コード'] = df_Tokyo_sample['市区町村コード'].astype(str).str.strip()

# 取引総額、建築年、建ぺい率、容積率を数値型に変換（エラーになる文字は強制的に欠損値 NaN にする）
df_Tokyo_sample['取引総額'] = pd.to_numeric(df_Tokyo_sample['取引総額'], errors='coerce')
df_Tokyo_sample['建築年'] = pd.to_numeric(df_Tokyo_sample['建築年'], errors='coerce').astype('Int64')
df_Tokyo_sample['建ぺい率'] = pd.to_numeric(df_Tokyo_sample['建ぺい率'], errors='coerce')
df_Tokyo_sample['容積率'] = pd.to_numeric(df_Tokyo_sample['容積率'], errors='coerce')

# テキストを数値化し、念のため欠損値(NaN)を考慮して整数型(Int64)にする
df_Tokyo_sample['駅徒歩分'] = pd.to_numeric(df_Tokyo_sample['駅徒歩分'], errors='coerce').astype('Int64')

# 分析のノイズになる「価格が0」または「価格が欠損」のデータを除外
df_Tokyo_sample = df_Tokyo_sample[df_Tokyo_sample['取引総額'] > 0]
df_Tokyo_sample = df_Tokyo_sample.dropna(subset=['取引総額'])

# インデックスをもう一度綺麗にリセット
df_Tokyo_sample = df_Tokyo_sample.reset_index(drop=True)

#軽量化したデータのダウンロード
output_path = KAGGLE_DATA_BASE / 'Kaggle_Tokyo_Residential_sample.csv'

df_Tokyo_sample.to_csv(output_path, index=False, encoding='utf-8-sig')


In [16]:
df_Tokyo_sample.shape

(4396, 7)

(4396, 7)